# Arctic Wolf Data Retrieval Service API - Interactive Investigation

This notebook provides a comprehensive guide to exploring and using the Arctic Wolf Data Retrieval Service API.

## Overview
- **API Version**: 1.0.0-beta
- **Endpoint**: Derived from the POD value in the .env file for your deployment region
- **Authentication**: PAK (Personal Access Key) Bearer Token
- **Python Version**: 3.9.6

## What You'll Learn
1. Setup and authentication
2. Discover available data sources
3. Explore predefined queries
4. Execute queries with different operators
5. Handle pagination and errors
6. Analyze results

## Resources
- [Customer Documentation](https://docs.arcticwolf.com/en/developer-and-oem/data-retrieval-api/arctic-wolf-data-retrieval-api)
)

## 1. Setup and Authentication

First, let's set up our environment and configure authentication.

In [ ]:
import requests
import json
from datetime import datetime, timedelta, timezone
from typing import Any, Optional, List, Dict
import pandas as pd
import time
import dotenv
import os
from pathlib import Path
from dataclasses import dataclass

from data_explorer_client import ApiConfig, prompt_for_organization_choice

# Load environment variables from .env
dotenv_path = Path.cwd() / ".env"
print(f"Loading environment from: {dotenv_path}")
dotenv.load_dotenv(dotenv_path, override=True)

# Load configuration. organization_id is intentionally NOT read from .env here -
# it's resolved from the PAK itself via the Organizations API, which avoids the
# classic copy/paste mistake of pasting a URL or UUID into ORGANIZATION_ID.
# Set ORGANIZATION_ID in .env only if this PAK has access to more than one
# organization and you want to skip the interactive prompt below.
required_vars = {
    "PAK_TOKEN": os.getenv("PAK_TOKEN"),
}
missing_vars = [name for name, value in required_vars.items() if not value]

if missing_vars:
    print("⚠️ Missing configuration values:")
    for name in missing_vars:
        print(f"  • {name}")
    print("\nSet the missing values in your .env file before running the API examples.")
    config = None
    client = None
else:
    config = ApiConfig.from_pak(
        pak_token=required_vars["PAK_TOKEN"],
        organization_id=os.getenv("ORGANIZATION_ID") or None,
        pod=os.getenv("POD") or None,
        on_multiple_organizations=prompt_for_organization_choice,
    )
    print("✅ Configuration resolved from PAK")
    print(repr(config))

In [ ]:
@dataclass
class DataExplorerError(Exception):
    """Exception for Data Retrieval API errors."""
    status_code: int
    message: str
    response_text: Optional[str] = None

    def __str__(self) -> str:
        parts = [f"HTTP {self.status_code}", self.message]
        if self.response_text and len(self.response_text) < 200:
            parts.append(f"response={self.response_text}")
        return " | ".join(parts)


def safe_api_call(
    func,
    *args,
    max_retries: int = 3,
    backoff_factor: float = 1.0,
    **kwargs
) -> Optional[Any]:
    """
    Wrapper for safe API calls with error handling and retry logic.
    Retries on transient errors (502, 503, etc) with exponential backoff.
    """
    for attempt in range(max_retries):
        try:
            return func(*args, **kwargs)
        except requests.exceptions.RequestException as e:
            if attempt < max_retries - 1:
                wait_time = backoff_factor * (2 ** attempt)
                print(f"⚠️ Attempt {attempt + 1} failed. Retrying in {wait_time:.1f}s...")
                time.sleep(wait_time)
            else:
                print(f"🌐 Network error after {max_retries} attempts")
                return None
        except json.JSONDecodeError as e:
            print(f"📄 JSON parsing error: {e}")
            return None
        except Exception as e:
            print(f"⚠️ Unexpected error: {e}")
            return None
    return None

print("✅ Error handling loaded")


## 2. Discover Data Sources

Let's explore what data sources are available and examine their schemas.

In [ ]:
from data_explorer_client import DataExplorerClient

if config is None:
    print("⚠️ Configuration is not available. Add the required values to your .env file first.")
    data_sources = None
    client = None
else:
    # Reuse the already-resolved config directly instead of rebuilding a new
    # ApiConfig here - that duplication is what let a bad organization_id slip
    # through unnoticed last time.
    client = DataExplorerClient(config)

    print("🔍 Discovering data sources...")
    data_sources = client.list_data_sources()

    if data_sources:
        print(f"\n📊 Available Data Sources ({len(data_sources)}):")
        for ds in data_sources:
            print(f"  • {ds['name']} (timeout: {ds['timeout']})")
    else:
        print("❌ No data sources found or error occurred")

In [ ]:
# Get schema for observations data source
if data_sources:
    print("🔍 Getting schema for 'observations' data source...")
    schema = client.get_data_source_schema("observations")
    
    if schema:
        print(f"\n📋 Observations Schema ({len(schema['fields'])} fields):")
        
        # Create a DataFrame for better display
        schema_df = pd.DataFrame([
            {
                'Field': field['name'],
                'Type': field['type'],
                'Nullable': field['nullable'],
                'Description': field.get('description', 'No description')[:50] + '...' if field.get('description', '') else 'No description'
            }
            for field in schema['fields'][:15]  # Show first 15 fields
        ])
        
        print(schema_df.to_string(index=False))
        
        if len(schema['fields']) > 15:
            print(f"\n... and {len(schema['fields']) - 15} more fields")
    else:
        print("❌ Could not retrieve schema")

## 3. Explore Predefined Queries

Now let's discover what predefined queries are available and examine their parameters.

In [ ]:
# Explore predefined queries through the client
if client is None:
    print("⚠️ The client is unavailable because configuration is missing.")
    queries = None
else:
    print("🔍 Discovering predefined queries...")
    queries = client.list_predefined_queries("observations")

    if queries:
        print(f"\n📝 Available Predefined Queries ({len(queries['queries'])}):")
        for i, query in enumerate(queries['queries'], 1):
            print(f"  {i}. {query['name']}")
            print(f"     {query['description']}")
            print()
    else:
        print("❌ No queries found or error occurred")

In [ ]:
if client is None:
    print("⚠️ The client is unavailable because configuration is missing.")
    queries = None
    ip_query_desc = None
else:
    if 'queries' not in globals() or queries is None:
        print("🔍 Discovering predefined queries...")
        queries = client.list_predefined_queries("observations")

    # Get detailed description of IP address query
    if queries:
        print("🔍 Getting detailed description for 'observations-by-ip-address' query...")
        ip_query_desc = client.describe_query("observations", "observations-by-ip-address")
    else:
        ip_query_desc = None

if ip_query_desc:
    print(f"\n📋 Query: {ip_query_desc['name']}")
    print(f"Description: {ip_query_desc['description']}")
    
    print(f"\n🔧 Parameters ({len(ip_query_desc['parameters'])}):")
    for param in ip_query_desc['parameters']:
        required = "✅ Required" if param['mandatory'] else "⚪ Optional"
        array_support = " (Array supported)" if param.get('canBeArray') else ""
        default = f" [Default: {param['default']}]" if param.get('default') is not None else ""
        
        print(f"  • {param['name']} ({param['type']}) - {required}{array_support}{default}")
        print(f"    {param['description']}")
        
        if param.get('allowed_operators'):
            print(f"    Operators: {', '.join(param['allowed_operators'])}")
        
        if param.get('validators'):
            validators = [v['type'] for v in param['validators']]
            print(f"    Validators: {', '.join(validators)}")
        print()
    
    # Show default columns
    if ip_query_desc.get('default_columns'):
        print(f"📊 Default Return Columns ({len(ip_query_desc['default_columns'])}):")
        for col in ip_query_desc['default_columns']:
            nullable = " (nullable)" if col.get('nullable') else ""
            print(f"  • {col['name']} ({col['type']}){nullable}")
            if col.get('description'):
                print(f"    {col.get('description')}")
else:
    print("❌ Could not retrieve query description")

## 4. Execute Queries

Now let's execute some queries with different parameters and operators.

In [ ]:
# Set up time range for queries (last 24 hours) using timezone-aware UTC datetimes
end_time = datetime.now(timezone.utc)
start_time = end_time - timedelta(hours=24)

print(f"🕐 Query time range:")
print(f"  Start: {start_time.strftime('%Y-%m-%d %H:%M:%S')} UTC")
print(f"  End:   {end_time.strftime('%Y-%m-%d %H:%M:%S')} UTC")

In [ ]:
# Example 1: Search by single IP address
print("🔍 Example 1: Single IP Address Search")
print("=" * 50)

if client is None:
    print("⚠️ Configure the environment first to run this query.")
else:
    parameters = [
        {
            "name": "start_time",
            "value": start_time.strftime("%Y-%m-%dT%H:%M:%SZ")
        },
        {
            "name": "end_time",
            "value": end_time.strftime("%Y-%m-%dT%H:%M:%SZ")
        },
        {
            "name": "ip_address",
            "comparisonOperator": "EQ", 
            "value": "10.171.170.112"
        }
    ]

    print(f"Searching for IP: 10.171.170.112")
    result = client.execute_query("observations", "observations-by-ip-address", parameters)

    if result:
        print(f"\n✅ Found {len(result['results'])} observations")
        
        if result['results']:
            # Convert to DataFrame for better display
            df = pd.DataFrame(result['results'], columns=[col['name'] for col in result['columns']])
            print(f"\n📊 Sample Results (showing first 5 rows):")
            print(df.head().to_string(index=False))
            
            print(f"\n📈 Column Info:")
            for col in result['columns']:
                print(f"  • {col['name']} ({col['type']})")
        else:
            print("No results found for this IP address")
    else:
        print("❌ Query execution failed")

In [ ]:
# Example 2: Search by multiple IP addresses using IN operator
print("🔍 Example 2: Multiple IP Addresses (IN operator)")
print("=" * 50)

if client is None:
    print("⚠️ Configure the environment first to run this query.")
else:
    # Build the parameters explicitly for this example so the cell can run independently
    multi_ip_parameters = [
        {
            "name": "start_time",
            "value": start_time.strftime("%Y-%m-%dT%H:%M:%SZ")
        },
        {
            "name": "end_time",
            "value": end_time.strftime("%Y-%m-%dT%H:%M:%SZ")
        },
        {
            "name": "ip_address",
            "comparisonOperator": "IN",
            "value": ["10.171.170.112", "10.171.170.105", "10.171.170.107"]
        }
    ]

    ip_list = multi_ip_parameters[2]["value"]
    print(f"Searching for IPs: {', '.join(ip_list)}")

    result = client.execute_query("observations", "observations-by-ip-address", multi_ip_parameters)

    if result:
        print(f"\n✅ Found {len(result['results'])} observations for multiple IPs")
        
        if result['results']:
            df = pd.DataFrame(result['results'], columns=[col['name'] for col in result['columns']])
            
            # Analyze results by IP if client.ip column exists
            if 'client.ip' in df.columns:
                ip_counts = df['client.ip'].value_counts()
                print(f"\n📊 Results by IP address:")
                for ip, count in ip_counts.items():
                    print(f"  • {ip}: {count} observations")
            
            print(f"\n📋 Sample Results:")
            print(df.head(3).to_string(index=False))
        else:
            print("No results found for these IP addresses")
    else:
        print("❌ Query execution failed")

In [ ]:
# Example 3: Search with CONTAINS operator (hostname)
print("🔍 Example 3: Hostname Contains Search")
print("=" * 50)

if client is None:
    print("⚠️ Configure the environment first to run this query.")
else:
    hostname_params = [
        {
            "name": "start_time",
            "value": start_time.strftime("%Y-%m-%dT%H:%M:%SZ")
        },
        {
            "name": "end_time",
            "value": end_time.strftime("%Y-%m-%dT%H:%M:%SZ")
        },
        {
            "name": "host_name",
            "comparisonOperator": "CONTAINS",
            "value": "DESKTOP"
        }
    ]

    print(f"Searching for hostnames containing: 'DESKTOP'")
    result = client.execute_query("observations", "observations-by-hostname", hostname_params)

    if result:
        print(f"\n✅ Found {len(result['results'])} observations with hostname containing 'DESKTOP'")
        
        if result['results']:
            df = pd.DataFrame(result['results'], columns=[col['name'] for col in result['columns']])
            print(f"\n📋 Sample Results:")
            print(df.head(3).to_string(index=False))
        else:
            print("No results found for hostnames containing 'DESKTOP'")
    else:
        print("❌ Query execution failed")

## 5. Advanced Usage & Best Practices

Let's explore advanced features like pagination, error handling, and performance monitoring.

In [ ]:
# Advanced query execution with error handling and pagination
def execute_query_with_pagination(data_source, query_id, parameters, limit=100, max_results=1000):
    """
    Execute query with automatic pagination to retrieve more results
    """
    all_results = []
    offset = 0
    columns = None
    
    print(f"🔄 Starting paginated query (limit={limit}, max_results={max_results})")

    if client is None:
        print("⚠️ Configure the environment first to run this query.")
        return {'columns': [], 'results': []}
    
    while len(all_results) < max_results:
        # Add pagination parameters
        paginated_params = parameters.copy()
        paginated_params.extend([
            {
                "name": "limit",
                "value": limit
            },
            {
                "name": "offset", 
                "value": offset
            }
        ])
        
        try:
            result = client.execute_query(data_source, query_id, paginated_params)
            
            if not result or not result['results']:
                print(f"📄 No more results at offset {offset}")
                break
                
            all_results.extend(result['results'])
            columns = result['columns']
            
            # If we got fewer results than the limit, we've reached the end
            if len(result['results']) < limit:
                print(f"📄 Reached end of results (got {len(result['results'])} < {limit})")
                break
                
            offset += limit
            print(f"📊 Retrieved {len(all_results)} results so far...")
            
        except Exception as e:
            print(f"❌ Error during pagination: {e}")
            break
    
    return {
        'columns': columns if columns else [],
        'results': all_results
    }

# Error handling wrapper
def safe_api_call(func, *args, **kwargs):
    """
    Wrapper for safe API calls with proper error handling
    """
    try:
        return func(*args, **kwargs)
    except requests.exceptions.RequestException as e:
        print(f"🌐 Network error: {e}")
        return None
    except json.JSONDecodeError as e:
        print(f"📄 JSON parsing error: {e}")
        return None
    except Exception as e:
        print(f"⚠️ Unexpected error: {e}")
        return None

print("✅ Advanced functions loaded")

In [ ]:
# Performance monitoring
def timed_query_execution(data_source, query_id, parameters):
    """
    Execute query with timing information
    """
    print(f"⏱️ Starting timed query execution...")
    start_time = time.time()

    if client is None:
        print("⚠️ Configure the environment first to run this query.")
        return None
    
    result = client.execute_query(data_source, query_id, parameters)
    
    end_time = time.time()
    
    if result:
        execution_time = end_time - start_time
        print(f"\n⏱️ Performance Metrics:")
        print(f"  • Execution time: {execution_time:.2f} seconds")
        print(f"  • Results returned: {len(result['results'])} rows")
        
        if result['results']:
            avg_time_per_row = execution_time / len(result['results'])
            print(f"  • Average time per row: {avg_time_per_row*1000:.2f} ms")
            
            # Estimate data size
            if result['columns']:
                estimated_size = len(result['results']) * len(result['columns']) * 50  # rough estimate
                print(f"  • Estimated data size: {estimated_size/1024:.2f} KB")
    
    return result

# Test performance with a simple query
print("🚀 Testing query performance...")
perf_result = timed_query_execution("observations", "observations-by-ip-address", parameters)

In [ ]:
# Data analysis helpers
def analyze_query_results(result):
    """
    Analyze query results and provide insights
    """
    if not result or not result['results']:
        print("❌ No results to analyze")
        return None
    
    df = pd.DataFrame(result['results'], columns=[col['name'] for col in result['columns']])
    
    print("📊 Query Results Analysis")
    print("=" * 30)
    print(f"📈 Dataset Overview:")
    print(f"  • Total rows: {len(df):,}")
    print(f"  • Total columns: {len(df.columns)}")
    print(f"  • Memory usage: {df.memory_usage(deep=True).sum() / 1024:.2f} KB")
    
    # Show data types
    print(f"\n🏷️ Column Types:")
    for col in result['columns']:
        sample_values = df[col['name']].dropna().head(3).tolist()
        sample_str = f" (e.g., {sample_values})" if sample_values else ""
        print(f"  • {col['name']}: {col['type']}{sample_str}")
    
    # Show sample data
    print(f"\n📋 Sample Data (first 3 rows):")
    print(df.head(3).to_string(index=False))
    
    # Basic statistics for numeric columns
    numeric_cols = df.select_dtypes(include=['number']).columns
    if len(numeric_cols) > 0:
        print(f"\n📊 Numeric Column Statistics:")
        print(df[numeric_cols].describe().round(2))
    
    return df

# Example usage of advanced pagination
print("🔍 Advanced Query Execution Example")
print("=" * 40)

# Use the advanced pagination function
large_result = execute_query_with_pagination(
    "observations", 
    "observations-by-ip-address", 
    parameters, 
    limit=50, 
    max_results=200
)

if large_result and large_result['results']:
    df = analyze_query_results(large_result)
    
    # Additional time-based analysis if timestamp column exists
    if df is not None and 'at_timestamp' in df.columns:
        try:
            df['at_timestamp'] = pd.to_datetime(df['at_timestamp'])
            print(f"\n🕐 Time Analysis:")
            print(f"  • Time range: {df['at_timestamp'].min()} to {df['at_timestamp'].max()}")
            print(f"  • Duration: {df['at_timestamp'].max() - df['at_timestamp'].min()}")
            
            # Group by hour
            hourly_counts = df.groupby(df['at_timestamp'].dt.hour).size()
            if len(hourly_counts) > 0:
                print(f"\n📅 Observations by hour:")
                for hour, count in hourly_counts.head(10).items():
                    print(f"  • Hour {hour:02d}: {count} observations")
        except Exception as e:
            print(f"⚠️ Could not analyze timestamps: {e}")
else:
    print("❌ No results from paginated query")

## 6. Experiment Zone

Use this section to experiment with different queries and parameters.

In [ ]:
# Experiment with different queries
# Try different query types available in your organization

print("🧪 Experiment Zone - Try Your Own Queries!")
print("=" * 50)

# Example: Query by user
user_params = [
    {
        "name": "start_time",
        "value": start_time.strftime("%Y-%m-%dT%H:%M:%SZ")
    },
    {
        "name": "end_time",
        "value": end_time.strftime("%Y-%m-%dT%H:%M:%SZ")
    },
    {
        "name": "user",
        "comparisonOperator": "CONTAINS",
        "value": "admin"  # Change this to a user you want to search for
    }
]

print("Trying user-based query...")
query_func = client.execute_query if client is not None else None
user_result = safe_api_call(query_func, "observations", "observations-by-user", user_params)

if user_result:
    print(f"✅ User query returned {len(user_result['results'])} results")
else:
    print("❌ User query failed or returned no results")

# Add your own experiments below:
# TODO: Try different query types
# TODO: Experiment with different operators
# TODO: Test different time ranges
# TODO: Try custom response_columns

In [ ]:
# Custom query builder helper
def build_query_parameters(start_time, end_time, **kwargs):
    """
    Helper function to build query parameters easily
    """
    params = [
        {
            "name": "start_time",
            "value": start_time.strftime("%Y-%m-%dT%H:%M:%SZ")
        },
        {
            "name": "end_time",
            "value": end_time.strftime("%Y-%m-%dT%H:%M:%SZ")
        }
    ]
    
    for param_name, (operator, value) in kwargs.items():
        params.append({
            "name": param_name,
            "comparisonOperator": operator,
            "value": value
        })
    
    return params

# Example usage of query builder
print("🛠️ Using Query Builder Helper")
print("=" * 30)

if "start_time" not in globals() or "end_time" not in globals():
    end_time = datetime.now(timezone.utc)
    start_time = end_time - timedelta(hours=24)

# Build a query for multiple domains
domain_params = build_query_parameters(
    start_time, 
    end_time,
    domain=("IN", ["example.com", "test.com", "demo.org"])
)

print("Built parameters for domain query:")
for param in domain_params:
    print(f"  • {param['name']}: {param.get('comparisonOperator', '')} {param['value']}")

# Try the domain query
query_func = client.execute_query if client is not None else None
domain_result = safe_api_call(query_func, "observations", "observations-by-domain", domain_params)

if domain_result:
    print(f"\n✅ Domain query returned {len(domain_result['results'])} results")
else:
    print("\n❌ Domain query failed or returned no results")

## Summary

This notebook has covered:

1. **Authentication** - Setting up PAK token authentication
2. **Discovery** - Finding available data sources and their schemas
3. **Query Exploration** - Understanding predefined queries and their parameters
4. **Query Execution** - Running queries with different operators (EQ, IN, CONTAINS)
5. **Advanced Features** - Pagination, error handling, and performance monitoring
6. **Data Analysis** - Processing and analyzing query results

### Next Steps

- Explore other available queries in your organization
- Experiment with different time ranges and parameters
- Build custom analysis workflows using the query results
- Integrate with other data analysis tools and workflows

### Resources

- [Arctic Wolf Data Retrieval API Documentation](https://docs.arcticwolf.com/en/developer-and-oem/data-retrieval-api/arctic-wolf-data-retrieval-api)

Happy querying! 🚀